# 04_tabular_baselines

Единый табличный пайплайн для Ridge, Lasso, Decision Tree и Random Forest. Ноутбук намеренно отделен от CatBoost, чтобы глобальные baseline-модели сравнивались по одному протоколу и на одинаковых признаках.


In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

DATA_DIR = Path("../data")
EXP_INFO_DIR = DATA_DIR / "experiment_info"

train_processed = pd.read_csv(DATA_DIR / "train_processed.csv", parse_dates=["date"])
test_processed = pd.read_csv(DATA_DIR / "test_processed.csv", parse_dates=["date"])
top_pairs_df = pd.read_csv(EXP_INFO_DIR / "top_pairs.csv")
with open(EXP_INFO_DIR / "splits.json", "r", encoding="utf-8") as f:
    splits_json = json.load(f)
splits = [{**s, "train_idx": pd.Index(s["train_idx"]), "val_idx": pd.Index(s["val_idx"])} for s in splits_json]
top_pairs = set(zip(top_pairs_df["store_nbr"], top_pairs_df["family"]))


## Признаковая политика

- `sales_sum` не используется из-за риска утечки.
- Для полного submission по умолчанию берется безопасный набор признаков, существующий и в train, и в test.
- Исследовательские расширения можно включать отдельно, но не смешивать их с production-like режимом.


In [ ]:
SAFE_NUMERIC_FEATURES = ["onpromotion"]
CATEGORY_FEATURES = ["store_nbr", "family", "city", "state", "store_type", "cluster", "type", "locale", "agg_description"]
BINARY_FEATURES = ["is_holiday", "is_payday", "transferred"]
TIME_FEATURES = ["dow", "weekofyear", "month", "day", "is_weekend"]


In [ ]:
def add_time_features(df):
    df = df.copy()
    df["dow"] = df["date"].dt.weekday.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["is_weekend"] = df["dow"].isin([5, 6]).astype("int8")
    return df

def add_lag_features(df, lags=(1, 7, 14, 28)):
    df = df.sort_values(["store_nbr", "family", "date"]).copy()
    for lag in lags:
        df[f"sales_lag_{lag}"] = df.groupby(["store_nbr", "family"])["sales"].shift(lag)
    return df

def add_rolling_features(df, windows=(7, 28)):
    df = df.sort_values(["store_nbr", "family", "date"]).copy()
    grouped = df.groupby(["store_nbr", "family"])["sales"]
    for window in windows:
        df[f"sales_rolling_mean_{window}"] = grouped.shift(1).rolling(window=window, min_periods=1).mean()
    return df

def filter_to_top_pairs(df):
    mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
    return df[mask].copy()


In [ ]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return {
        "RMSLE": np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2)),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "WAPE": np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true)),
    }

def make_train_val_for_split(split, use_top_pairs=False, lags=(1, 7, 14, 28), windows=(7, 28)):
    train_df = train_processed.loc[split["train_idx"]].copy()
    val_df = train_processed.loc[split["val_idx"]].copy()
    if use_top_pairs:
        train_df = filter_to_top_pairs(train_df)
        val_df = filter_to_top_pairs(val_df)
    combined = pd.concat([train_df, val_df], axis=0).sort_values(["store_nbr", "family", "date"])
    combined = add_time_features(combined)
    combined = add_lag_features(combined, lags=lags)
    combined = add_rolling_features(combined, windows=windows)
    lag_roll_cols = [c for c in combined.columns if c.startswith("sales_lag_") or c.startswith("sales_rolling_mean_")]
    combined = combined.dropna(subset=lag_roll_cols)
    train_mask = combined["date"] <= train_df["date"].max()
    val_mask = (combined["date"] >= val_df["date"].min()) & (combined["date"] <= val_df["date"].max())
    train_fe = combined.loc[train_mask].copy()
    val_fe = combined.loc[val_mask].copy()
    feature_cols = SAFE_NUMERIC_FEATURES + BINARY_FEATURES + TIME_FEATURES + lag_roll_cols + CATEGORY_FEATURES
    feature_cols = [c for c in feature_cols if c in train_fe.columns and c in val_fe.columns]
    return train_fe, val_fe, feature_cols


In [ ]:
def make_pipeline(model, feature_cols, train_fe):
    numeric_cols = [c for c in feature_cols if c in (SAFE_NUMERIC_FEATURES + TIME_FEATURES) or c.startswith("sales_lag_") or c.startswith("sales_rolling_mean_")]
    binary_cols = [c for c in feature_cols if c in BINARY_FEATURES]
    categorical_cols = [c for c in feature_cols if c in CATEGORY_FEATURES]
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([("scaler", StandardScaler())]), numeric_cols),
            ("bin", "passthrough", binary_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ]
    )
    return Pipeline([("prep", preprocessor), ("model", model)])

MODEL_REGISTRY = {
    "ridge": Ridge(alpha=1.0),
    "lasso": Lasso(alpha=0.0005, max_iter=5000),
    "decision_tree": DecisionTreeRegressor(max_depth=12, random_state=42),
    "random_forest": RandomForestRegressor(n_estimators=150, max_depth=14, random_state=42, n_jobs=-1),
}


In [ ]:
def evaluate_model(model_name, use_top_pairs=False):
    rows = []
    for split in splits:
        train_fe, val_fe, feature_cols = make_train_val_for_split(split, use_top_pairs=use_top_pairs)
        pipeline = make_pipeline(MODEL_REGISTRY[model_name], feature_cols, train_fe)
        X_train, y_train = train_fe[feature_cols], train_fe["sales"]
        X_val, y_val = val_fe[feature_cols], val_fe["sales"]
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_val)
        metrics = calculate_metrics(y_val, y_pred)
        metrics.update({"split": split["name"], "model": model_name, "scope": "top_pairs" if use_top_pairs else "all_series"})
        rows.append(metrics)
    return pd.DataFrame(rows)


## Пример запуска

Ниже оставлен минимальный пример. Полный прогон по всем моделям лучше делать поочередно, с сохранением CSV в `data/experiment_info/`.


In [ ]:
# example_metrics = evaluate_model("ridge", use_top_pairs=False)
# example_metrics.to_csv(EXP_INFO_DIR / "ridge_metrics_all_series.csv", index=False)
# example_metrics.head()
